# A10. Why Medellín's raw PM2.5 components are excluded from the candidate pool (leakage check)

Medellín's `pm25` target is built from two ingredients: the Aburrá Valley (Antioquia) network mean and the Hourmean network
mean as a calibrated fallback. The raw ingredient columns (`pm25_antioquia`, `pm25_hourmean`) must not be candidates: a neural net given
one of them alone "rediscovers" the target. The first Medellín selection run after the Antioquia merge (v11 rebuild, MASTER.md) picked
`pm25_hourmean` first, at pooled LOYO R2 = 0.856. The raw output of that run was not saved (the notebook was re-run with the exclusion),
so this notebook repeats the one-variable test on the current master. Scoring: the same neural net as feature selection, leave-one-year-out, pooled R2.

In [1]:
# Setup
import sys
from pathlib import Path
HERE = Path.cwd().resolve()
sys.path.insert(0, str(HERE.parent)); sys.path.insert(0, str(HERE.parents[1]))
import pandas as pd
from style import ROOT, DATA_OUT
from common.feature_selection import loyo_r2

In [2]:
# Settings (edit here)
COLUMNS = ["pm25_hourmean", "pm25_antioquia"]
SAVE = True

In [3]:
# Analysis
d = pd.read_parquet(ROOT / "Data/processed/colombia/medellin/medellin_master_dataset.parquet")
d["date"] = pd.to_datetime(d["date"]); d["year"] = d["date"].dt.year
rows = []
for c in COLUMNS:
    r = d.dropna(subset=[c, "pm25"]).reset_index(drop=True)
    rows.append(dict(column=c, n_days=len(r), pearson_r=float(r[c].corr(r.pm25)), loyo_pooled_r2_one_variable=float(loyo_r2(r, [c], "pm25"))))
out = pd.DataFrame(rows); print(out.round(3).to_string(index=False))
if SAVE: out.to_csv(DATA_OUT / "medellin_pm25_leakage_check.csv", index=False)

        column  n_days  pearson_r  loyo_pooled_r2_one_variable
 pm25_hourmean    2874      0.931                        0.856
pm25_antioquia    2372      1.000                        0.998
